# Pokémon Gen 2 OU — Reproducible battle-outcome ensemble

The maintained pipeline combines a frozen baseline with three unanimous experts. All features come from the competition files. The 70.17% exploratory OOF figure is not a confirmed Kaggle score.


## Clone the repository and install dependencies

Internet access is required only to clone GitHub, not as a modeling input.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/fds-pokemon")
REPO_URL = "https://github.com/lucamartone/fds-pokemon.git"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise ValueError(f"Not a Git repository: {REPO_DIR}")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)


## Find official files and generate the submission


In [ ]:
train_files = list(Path("/kaggle/input").rglob("train.jsonl"))
if len(train_files) != 1:
    raise FileNotFoundError(f"Expected one train.jsonl; found {len(train_files)}")
DATA_DIR = train_files[0].parent
for name in ("test.jsonl", "type_chart_gen2.csv", "sample_submission.csv"):
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUTPUT = Path("/kaggle/working/submission.csv")
subprocess.run([sys.executable, str(REPO_DIR / "run.py"), "--data", str(DATA_DIR), "--output", str(OUTPUT)], cwd=REPO_DIR, check=True)
print("Submission:", OUTPUT, "Commit:", COMMIT)


In [ ]:
import pandas as pd
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
submission = pd.read_csv(OUTPUT)
assert submission.shape == sample.shape
assert submission.columns.equals(sample.columns)
assert submission.battle_id.equals(sample.battle_id)
assert set(submission.player_won.unique()).issubset({0, 1})
display(submission.head())


The competition also requires two other selected final submissions. This notebook reproduces the maintained consensus model only; additional selected submissions must be reproducibly implemented before final delivery.
